In [3]:
from dbrepo.RestClient import RestClient

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="gokayyil",
    password="123123Go."
)

print(client.whoami())

gokayyil
gokayyil


In [4]:
DATABASE_ID = "a6229b2e-5a3d-4a46-93aa-e9474f492ec1"

database = client.get_database(DATABASE_ID)

print(database)

id='a6229b2e-5a3d-4a46-93aa-e9474f492ec1' name='coffee_productivity_prediction' exchange_name='dbrepo' internal_name='coffee_productivity_prediction_wbmm' is_public=False is_schema_public=False is_dashboard_enabled=False container=ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None) owner=UserBrief(username='gokayyil', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) contact=UserBrief(username='gokayyil', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) identifiers=[] subsets=[] tables=[Table(id='9abc69b3-932e-4a20-93a6-9e3cb4ac9676', database_id='a6229b2e-5a3d-4a46-93aa-e9474f492ec1', name='health_metrics', owner=UserBrief(username='gokayyil', id=None, name=None, orcid=None, qualified_name=None, given_name=Non

In [3]:
[m for m in dir(client) if "table" in m.lower()]

['analyse_table_statistics',
 'create_table',
 'create_table_data',
 'delete_table',
 'delete_table_data',
 'get_table',
 'get_table_data',
 'get_table_data_count',
 'get_table_history',
 'get_tables',
 'import_table_data',
 'update_table_column',
 'update_table_data',
 'update_table_statistics']

In [5]:
import inspect

print(inspect.signature(client.create_table))

(database_id: str, name: str, is_public: bool, is_schema_public: bool, dataframe: pandas.DataFrame, description: str = None, with_data: bool = True) -> dbrepo.api.dto.TableBrief


In [6]:
import pandas as pd
from pathlib import Path

processed_dir = Path("../data/processed")

table_files = {
    "countries": "countries.csv",
    "genders": "genders.csv",
    "occupations": "occupations.csv",
    "participants": "participants.csv",
    "coffee_consumption": "coffee_consumption.csv",
    "sleep_metrics": "sleep_metrics.csv",
    "health_metrics": "health_metrics.csv",
    "lifestyle_metrics": "lifestyle_metrics.csv"
}

for table_name, filename in table_files.items():
    df_table = pd.read_csv(processed_dir / filename)
    print(table_name, df_table.shape)
    display(df_table.head())

countries (20, 2)


,country_id,country_name
0,1,Australia
1,2,Belgium
2,3,Brazil
3,4,Canada
4,5,China


genders (3, 2)


,gender_id,gender_name
0,1,Female
1,2,Male
2,3,Other


occupations (5, 2)


...

,participant_id,sleep_hours,sleep_quality
0,1,7.5,Good
1,2,6.2,Good
2,3,5.9,Fair
3,4,7.3,Good
4,5,5.3,Fair


health_metrics (10000, 4)


,participant_id,bmi,heart_rate_bpm,health_issues
0,1,24.9,78,NaN
1,2,20.0,67,NaN
2,3,22.7,59,Mild
3,4,24.7,71,Mild
4,5,24.1,76,Mild


lifestyle_metrics (10000, 5)


,participant_id,stress_level,physical_activity_hours,smoking,alcohol_consumption
0,1,Low,14.5,False,False
1,2,Low,11.0,False,False
2,3,Medium,11.2,False,False
3,4,Low,6.6,False,False
4,5,Medium,8.5,False,True


In [8]:
primary_keys = {
    "countries": "country_id",
    "genders": "gender_id",
    "occupations": "occupation_id",
    "participants": "participant_id",
    "coffee_consumption": "participant_id",
    "sleep_metrics": "participant_id",
    "health_metrics": "participant_id",
    "lifestyle_metrics": "participant_id"
}

created_tables = {}

for table_name, filename in table_files.items():
    df_table = pd.read_csv(processed_dir / filename)

    if table_name == "health_metrics":
        df_table["health_issues"] = df_table["health_issues"].fillna("None")

    pk = primary_keys[table_name]
    df_table = df_table.set_index(pk)

    print(f"Creating table: {table_name} with shape {df_table.shape}, primary key: {pk}")

    created_table = client.create_table(
        database_id=DATABASE_ID,
        name=table_name,
        is_public=True,
        is_schema_public=True,
        dataframe=df_table,
        description=table_descriptions[table_name],
        with_data=True
    )

    created_tables[table_name] = created_table

    print(created_table)

Creating table: countries with shape (20, 1), primary key: country_id
2026-05-26 15:16:03,331 root         WARNING default to 'text' for column country_name and type <class 'numpy.dtype'>


ResponseCodeError: Failed to upload: response code: 500 is not 201 (CREATED) or 204 (NO CONTENT): {"timestamp":"2026-05-26T13:16:44.595+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/upload"}

In [9]:
tables = client.get_tables(DATABASE_ID)

for table in tables:
    print(table.id, table.name)

018641cc-175b-4fff-b05c-b5afe4d9733c countries


In [10]:
created_tables = {
    "countries": client.get_table(
        database_id=DATABASE_ID,
        table_id="018641cc-175b-4fff-b05c-b5afe4d9733c"
    )
}

for table_name, filename in table_files.items():
    if table_name == "countries":
        continue

    df_table = pd.read_csv(processed_dir / filename)

    if table_name == "health_metrics":
        df_table["health_issues"] = df_table["health_issues"].fillna("None")

    pk = primary_keys[table_name]
    df_table = df_table.set_index(pk)

    print(f"Creating schema only table: {table_name}")

    created_table = client.create_table(
        database_id=DATABASE_ID,
        name=table_name,
        is_public=True,
        is_schema_public=True,
        dataframe=df_table,
        description=table_descriptions[table_name],
        with_data=False
    )

    created_tables[table_name] = created_table
    print(created_table.id, created_table.name)

Creating schema only table: genders
2026-05-26 15:18:43,995 root         WARNING default to 'text' for column gender_name and type <class 'numpy.dtype'>
e5980c01-dacf-481a-84f8-1f83bd84095c genders
Creating schema only table: occupations
2026-05-26 15:18:44,406 root         WARNING default to 'text' for column occupation_name and type <class 'numpy.dtype'>
8a73da1b-c5b1-4666-968c-ac1d47dbcbc7 occupations
Creating schema only table: participants
913c5c1d-3e46-4393-9e5e-744ab04e0ad2 participants
Creating schema only table: coffee_consumption
6e828acb-4698-4f34-9a62-1a753b768554 coffee_consumption
Creating schema only table: sleep_metrics
2026-05-26 15:18:45,625 root         WARNING default to 'text' for column sleep_quality and type <class 'numpy.dtype'>
83a691e2-a7ad-4b5a-b468-03c3d63d727b sleep_metrics
Creating schema only table: health_metrics
2026-05-26 15:18:45,972 root         WARNING default to 'text' for column health_issues and type <class 'numpy.dtype'>
9abc69b3-932e-4a20-93a6-

In [11]:
countries_df = pd.read_csv(processed_dir / "countries.csv")

countries_table_id = "018641cc-175b-4fff-b05c-b5afe4d9733c"

client.import_table_data(
    database_id=DATABASE_ID,
    table_id=countries_table_id,
    dataframe=countries_df
)

print("countries import requested")

ResponseCodeError: Failed to upload: response code: 500 is not 201 (CREATED) or 204 (NO CONTENT): {"timestamp":"2026-05-26T13:20:48.607+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/upload"}

In [12]:
import inspect
print(inspect.signature(client.create_table_data))

(database_id: str, table_id: str, data: dict) -> None


In [13]:
countries_df = pd.read_csv(processed_dir / "countries.csv")

first_row = countries_df.iloc[0].to_dict()
print(first_row)

client.create_table_data(
    database_id=DATABASE_ID,
    table_id="018641cc-175b-4fff-b05c-b5afe4d9733c",
    data=first_row
)

print("Inserted first country row")

{'country_id': 1, 'country_name': 'Australia'}
Inserted first country row


In [14]:
countries_df = pd.read_csv(processed_dir / "countries.csv")

for _, row in countries_df.iloc[1:].iterrows():
    client.create_table_data(
        database_id=DATABASE_ID,
        table_id="018641cc-175b-4fff-b05c-b5afe4d9733c",
        data=row.to_dict()
    )

print("Inserted remaining countries")

Inserted remaining countries


In [15]:
count = client.get_table_data_count(
    database_id=DATABASE_ID,
    table_id="018641cc-175b-4fff-b05c-b5afe4d9733c"
)

print(count)

20


In [16]:
table_ids = {
    "countries": "018641cc-175b-4fff-b05c-b5afe4d9733c",
    "genders": "e5980c01-dacf-481a-84f8-1f83bd84095c",
    "occupations": "8a73da1b-c5b1-4666-968c-ac1d47dbcbc7"
}

for table_name in ["genders", "occupations"]:

    df = pd.read_csv(processed_dir / f"{table_name}.csv")

    for _, row in df.iterrows():
        client.create_table_data(
            database_id=DATABASE_ID,
            table_id=table_ids[table_name],
            data=row.to_dict()
        )

    count = client.get_table_data_count(
        database_id=DATABASE_ID,
        table_id=table_ids[table_name]
    )

    print(table_name, count)

genders 3
occupations 5


In [29]:
INTEGER_COLUMNS = {
    "country_id",
    "gender_id",
    "occupation_id",
    "participant_id",
    "age",
    "heart_rate_bpm"
}

DECIMAL_COLUMNS = {
    "coffee_intake_cups_per_day",
    "caffeine_mg_per_day",
    "sleep_hours",
    "bmi",
    "physical_activity_hours"
}

BOOLEAN_COLUMNS = {
    "smoking",
    "alcohol_consumption"
}


def clean_value_by_column(column, value):
    if pd.isna(value):
        return None

    if column in INTEGER_COLUMNS:
        return int(value)

    if column in DECIMAL_COLUMNS:
        return str(float(value))

    if column in BOOLEAN_COLUMNS:
        return bool(value)

    return str(value)


def row_to_clean_dict(row):
    return {
        column: clean_value_by_column(column, value)
        for column, value in row.to_dict().items()
    }


def insert_dataframe_rows(database_id, table_id, dataframe, table_name):
    total_rows = len(dataframe)

    for index, (_, row) in enumerate(dataframe.iterrows(), start=1):
        data = row_to_clean_dict(row)

        client.create_table_data(
            database_id=database_id,
            table_id=table_id,
            data=data
        )

        if index % 100 == 0 or index == total_rows:
            print(f"{table_name}: inserted {index}/{total_rows} rows")

    print(f"{table_name}: completed")

In [18]:
participants_sample = pd.read_csv(
    processed_dir / "participants.csv"
).head(100)

insert_dataframe_rows(
    DATABASE_ID,
    "913c5c1d-3e46-4393-9e5e-744ab04e0ad2",
    participants_sample,
    "participants_sample"
)

participants_sample: inserted 100/100 rows
participants_sample: completed


In [19]:
client.get_table_data_count(
    DATABASE_ID,
    "913c5c1d-3e46-4393-9e5e-744ab04e0ad2"
)

100

In [20]:
table_ids.update({
    "participants": "913c5c1d-3e46-4393-9e5e-744ab04e0ad2",
    "coffee_consumption": "6e828acb-4698-4f34-9a62-1a753b768554",
    "sleep_metrics": "83a691e2-a7ad-4b5a-b468-03c3d63d727b",
    "health_metrics": "9abc69b3-932e-4a20-93a6-9e3cb4ac9676",
    "lifestyle_metrics": "dddb6b09-ba3f-4003-bf93-73786410e886"
})

for table_name in [
    "coffee_consumption",
    "sleep_metrics",
    "health_metrics",
    "lifestyle_metrics"
]:
    df = pd.read_csv(processed_dir / f"{table_name}.csv").head(100)

    if table_name == "health_metrics":
        df["health_issues"] = df["health_issues"].fillna("None")

    insert_dataframe_rows(
        DATABASE_ID,
        table_ids[table_name],
        df,
        table_name
    )

    count = client.get_table_data_count(
        DATABASE_ID,
        table_ids[table_name]
    )

    print(table_name, count)

ResponseCodeError: Failed to insert table data: response code: 500 is not 201 (CREATED): {"timestamp":"2026-05-26T13:27:20.096+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/database/a6229b2e-5a3d-4a46-93aa-e9474f492ec1/table/6e828acb-4698-4f34-9a62-1a753b768554/data"}

In [22]:
coffee_df = pd.read_csv(processed_dir / "coffee_consumption.csv").head(1)

print(row_to_clean_dict(coffee_df.iloc[0]))

client.create_table_data(
    database_id=DATABASE_ID,
    table_id=table_ids["coffee_consumption"],
    data=row_to_clean_dict(coffee_df.iloc[0])
)

print("Inserted first coffee row")

{'participant_id': 1.0, 'coffee_intake_cups_per_day': 3.5, 'caffeine_mg_per_day': 328.1}


ResponseCodeError: Failed to insert table data: response code: 500 is not 201 (CREATED): {"timestamp":"2026-05-26T13:29:24.465+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/database/a6229b2e-5a3d-4a46-93aa-e9474f492ec1/table/6e828acb-4698-4f34-9a62-1a753b768554/data"}

In [23]:
coffee_table = client.get_table(
    database_id=DATABASE_ID,
    table_id=table_ids["coffee_consumption"]
)

print(coffee_table)

id='6e828acb-4698-4f34-9a62-1a753b768554' database_id='a6229b2e-5a3d-4a46-93aa-e9474f492ec1' name='coffee_consumption' owner=UserBrief(username='gokayyil', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='ea78c4ce-44da-4468-b4d7-3880baf9b1e1', name='participant_id', database_id='a6229b2e-5a3d-4a46-93aa-e9474f492ec1', table_id='6e828acb-4698-4f34-9a62-1a753b768554', ord=0, internal_name='participant_id', is_null_allowed=False, type=<ColumnType.BIGINT: 'bigint'>, alias=None, description=None, size=None, d=None, mean=None, median=None, concept=None, unit=None, concept_uri=None, unit_uri=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='d3cf2175-5393-43d6-8707-31a39be698b4', name='coffee_intake_cups_per_day', database_id='a6229b2e-5a3d-4a46-93aa-e9474f492ec1', table_id='6e828acb-4698-4f34-9a62-1a753b768554', ord=1, in

In [24]:
for column in coffee_table.columns:
    print(column)

id='ea78c4ce-44da-4468-b4d7-3880baf9b1e1' name='participant_id' database_id='a6229b2e-5a3d-4a46-93aa-e9474f492ec1' table_id='6e828acb-4698-4f34-9a62-1a753b768554' ord=0 internal_name='participant_id' is_null_allowed=False type=<ColumnType.BIGINT: 'bigint'> alias=None description=None size=None d=None mean=None median=None concept=None unit=None concept_uri=None unit_uri=None enums=[] sets=[] index_length=None length=None data_length=None max_data_length=None num_rows=None val_min=None val_max=None std_dev=None
id='d3cf2175-5393-43d6-8707-31a39be698b4' name='coffee_intake_cups_per_day' database_id='a6229b2e-5a3d-4a46-93aa-e9474f492ec1' table_id='6e828acb-4698-4f34-9a62-1a753b768554' ord=1 internal_name='coffee_intake_cups_per_day' is_null_allowed=False type=<ColumnType.DECIMAL: 'decimal'> alias=None description=None size=40 d=20 mean=None median=None concept=None unit=None concept_uri=None unit_uri=None enums=[] sets=[] index_length=None length=None data_length=None max_data_length=None

In [25]:
coffee_row = {
    "participant_id": 1,
    "coffee_intake_cups_per_day": "3.5",
    "caffeine_mg_per_day": "328.1"
}

client.create_table_data(
    database_id=DATABASE_ID,
    table_id=table_ids["coffee_consumption"],
    data=coffee_row
)

print("Inserted coffee row with decimals as strings")

Inserted coffee row with decimals as strings


In [28]:
coffee_df = pd.read_csv(processed_dir / "coffee_consumption.csv").iloc[1:100]

insert_dataframe_rows(
    DATABASE_ID,
    table_ids["coffee_consumption"],
    coffee_df,
    "coffee_consumption"
)

count = client.get_table_data_count(
    DATABASE_ID,
    table_ids["coffee_consumption"]
)

print(count)

ResponseCodeError: Failed to insert table data: response code: 500 is not 201 (CREATED): {"timestamp":"2026-05-26T13:32:48.683+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/database/a6229b2e-5a3d-4a46-93aa-e9474f492ec1/table/6e828acb-4698-4f34-9a62-1a753b768554/data"}

In [30]:
coffee_df = pd.read_csv(processed_dir / "coffee_consumption.csv")

test_row = coffee_df.iloc[1]

print(row_to_clean_dict(test_row))

{'participant_id': 2, 'coffee_intake_cups_per_day': '1.0', 'caffeine_mg_per_day': '94.1'}


In [31]:
client.create_table_data(
    database_id=DATABASE_ID,
    table_id=table_ids["coffee_consumption"],
    data={
        "participant_id": 2,
        "coffee_intake_cups_per_day": "1.0",
        "caffeine_mg_per_day": "94.1"
    }
)

print("Inserted coffee row 2 manually")

Inserted coffee row 2 manually


In [32]:
client.get_table_data_count(DATABASE_ID, table_ids["coffee_consumption"])

2

In [33]:
coffee_df = pd.read_csv(
    processed_dir / "coffee_consumption.csv"
).iloc[2:100]

insert_dataframe_rows(
    DATABASE_ID,
    table_ids["coffee_consumption"],
    coffee_df,
    "coffee_consumption"
)

count = client.get_table_data_count(
    DATABASE_ID,
    table_ids["coffee_consumption"]
)

print(count)

coffee_consumption: inserted 98/98 rows
coffee_consumption: completed
100


In [34]:
for table_name in [
    "sleep_metrics",
    "health_metrics",
    "lifestyle_metrics"
]:
    df = pd.read_csv(processed_dir / f"{table_name}.csv").head(100)

    if table_name == "health_metrics":
        df["health_issues"] = df["health_issues"].fillna("None")

    insert_dataframe_rows(
        DATABASE_ID,
        table_ids[table_name],
        df,
        table_name
    )

    count = client.get_table_data_count(
        DATABASE_ID,
        table_ids[table_name]
    )

    print(table_name, count)

sleep_metrics: inserted 100/100 rows
sleep_metrics: completed
sleep_metrics 100
health_metrics: inserted 100/100 rows
health_metrics: completed
health_metrics 100
lifestyle_metrics: inserted 100/100 rows
lifestyle_metrics: completed
lifestyle_metrics 100


In [35]:
[m for m in dir(client) if "view" in m.lower()]

['create_view',
 'delete_view',
 'get_view',
 'get_view_data',
 'get_view_data_count',
 'get_views',
 'update_view']

In [36]:
[m for m in dir(client) if "create" in m.lower()]

['create_container',
 'create_database',
 'create_database_access',
 'create_identifier',
 'create_subset',
 'create_table',
 'create_table_data',
 'create_view']

In [37]:
import inspect

for method_name in dir(client):
    if "view" in method_name.lower():
        print(method_name)

create_view
delete_view
get_view
get_view_data
get_view_data_count
get_views
update_view


In [38]:
import inspect
print(inspect.signature(client.create_view))

(database_id: str, name: str, query: dbrepo.api.dto.QueryDefinition, is_public: bool, is_schema_public: bool) -> dbrepo.api.dto.ViewBrief


In [39]:
from dbrepo.api.dto import QueryDefinition

help(QueryDefinition)

Help on class QueryDefinition in module dbrepo.api.dto:

class QueryDefinition(pydantic.main.BaseModel)
 |  QueryDefinition(**data: 'Any') -> 'None'
 |
 |  Method resolution order:
 |      QueryDefinition
 |      pydantic.main.BaseModel
 |      builtins.object
 |
 |  Data descriptors defined here:
 |
 |  __weakref__
 |      list of weak references to the object
 |
 |  ----------------------------------------------------------------------
 |  Data and other attributes defined here:
 |
 |  __abstractmethods__ = frozenset()
 |
 |  __annotations__ = {'columns': 'List[str]', 'datasources': 'List[str]',...
 |
 |  __class_vars__ = set()
 |
 |  __private_attributes__ = {}
 |
 |  __pydantic_complete__ = True
 |
 |  __pydantic_computed_fields__ = {}
 |
 |  __pydantic_core_schema__ = {'cls': <class 'dbrepo.api.dto.QueryDefinit...
 |
 |  __pydantic_custom_init__ = False
 |
 |  __pydantic_decorators__ = DecoratorInfos(validators={}, field_validato...
 |
 |  __pydantic_extra_info__ = None
 |
 |  __p

In [40]:
import inspect
print(inspect.signature(QueryDefinition))

(*, columns: List[str], datasources: List[str], joins: Optional[List[dbrepo.api.dto.JoinDefinition]] = None, filters: Optional[List[dbrepo.api.dto.FilterDefinition]] = None, orders: Optional[List[dbrepo.api.dto.OrderDefinition]] = None) -> None


In [41]:
from dbrepo.api.dto import JoinDefinition
import inspect

print(inspect.signature(JoinDefinition))
print(JoinDefinition.model_fields)

(**data: 'Any') -> 'None'
{'type': FieldInfo(annotation=ForwardRef('JoinType'), required=True), 'datasource': FieldInfo(annotation=str, required=True), 'conditionals': FieldInfo(annotation=ForwardRef('List[ConditionalDefinition]'), required=True)}


In [42]:
from dbrepo.api.dto import ConditionalDefinition
import inspect

print(inspect.signature(ConditionalDefinition))
print(ConditionalDefinition.model_fields)

(*, column: str, foreign_column: str) -> None
{'column': FieldInfo(annotation=str, required=True), 'foreign_column': FieldInfo(annotation=str, required=True)}


In [43]:
from dbrepo.api.dto import JoinType

print(list(JoinType))

[<JoinType.INNER: 'inner'>, <JoinType.LEFT: 'left'>, <JoinType.RIGHT: 'right'>, <JoinType.CROSS: 'cross'>]


In [44]:
from dbrepo.api.dto import QueryDefinition, JoinDefinition, ConditionalDefinition, JoinType

test_query = QueryDefinition(
    columns=[
        "participants.participant_id",
        "participants.age",
        "genders.gender_name"
    ],
    datasources=[
        "participants"
    ],
    joins=[
        JoinDefinition(
            type=JoinType.INNER,
            datasource="genders",
            conditionals=[
                ConditionalDefinition(
                    column="participants.gender_id",
                    foreign_column="genders.gender_id"
                )
            ]
        )
    ]
)

test_view = client.create_view(
    database_id=DATABASE_ID,
    name="vw_participants_gender_test",
    query=test_query,
    is_public=True,
    is_schema_public=True
)

print(test_view)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [45]:
database = client.get_database(DATABASE_ID)

print("tables:", len(database.tables))
for table in database.tables:
    print(table.name, table.internal_name, len(table.columns))

tables: 8
health_metrics health_metrics 4
lifestyle_metrics lifestyle_metrics 5
coffee_consumption coffee_consumption 3
sleep_metrics sleep_metrics 3
participants participants 5
occupations occupations 2
genders genders 2
countries countries 2


In [46]:
views = client.get_views(DATABASE_ID)

for view in views:
    print(view.id, view.name)

49246fd3-4515-48c1-b23a-fe183deea81c vw_productivity_features


In [47]:
VIEW_ID = "49246fd3-4515-48c1-b23a-fe183deea81c"

view_count = client.get_view_data_count(
    database_id=DATABASE_ID,
    view_id=VIEW_ID
)

print(view_count)

100


In [48]:
view_data = client.get_view_data(
    database_id=DATABASE_ID,
    view_id=VIEW_ID
)

print(view_data[:5])

MalformedError: Failed to get view data: {"status":"BAD_REQUEST","message":"Malformed query: [TASK_WRITE_FAILED] Task failed while writing rows to s3a://dbrepo/ec0691b7-c046-436e-aefa-1d819d76239f. SQLSTATE: 58030","code":"error.query.invalid"}

In [49]:
view_data = client.get_view_data(
    database_id=DATABASE_ID,
    view_id=VIEW_ID,
    page=0,
    size=5
)

display(view_data)

MalformedError: Failed to get view data: {"status":"BAD_REQUEST","message":"Malformed query: [TASK_WRITE_FAILED] Task failed while writing rows to s3a://dbrepo/e6a7d227-ed6b-4b2e-aee3-768def4be705. SQLSTATE: 58030","code":"error.query.invalid"}

## DBRepo View Verification Note

The DBRepo view `vw_productivity_features` was created successfully and `get_view_data_count` returned 100 rows.

Retrieving the view rows through `get_view_data` failed with a DBRepo service side error:

`TASK_WRITE_FAILED ... writing rows to s3a://dbrepo`

This indicates that the view query is valid, but the DBRepo test instance failed while exporting or materializing the result rows. The count endpoint was therefore used as the verification step.

## Final DBRepo Limitation and Subset Decision

Because DBRepo returned server side errors during bulk upload and view row retrieval, the experiment was continued with a DBRepo validation subset of 100 rows. This decision follows the course forum guidance where lecturers confirmed that continuing with a subset is acceptable when DBRepo fails due to server side limitations.

The DBRepo schema, row based data loading, and view count verification were successfully demonstrated. The complete local experiment remains available in the repository using the full processed dataset.